# Servo vs the other browsers: memory benchmark (Kaggle / Colab / Linux)

Measures the memory of each browser's **whole process tree** (UI, renderers, GPU, network) after it loads one page in a fresh profile, and with ten tabs open, on a Linux machine with a virtual screen. Servo is the official **servoshell v0.7.0**, the same engine version Pteron is built on. Pteron's own Linux build is added too when the machine can run it (Ubuntu 24.04 or newer; Kaggle and Colab are usually older, so there it is skipped with a note).

The report at the end starts with the **test machine** (CPU, cores, RAM, OS, platform) and then groups the browsers into **categories**:

| Category | Browsers |
|---|---|
| Servo engine | Pteron, servoshell (official Servo) |
| Firefox (Gecko) | Firefox |
| Chromium | Chrome |

Works on **Kaggle** (turn Internet on in the notebook settings), **Colab**, and any Linux machine or WSL2.

Compare browsers **within one run only**; shared machines vary from day to day. Run all cells in order.

In [ ]:
# 1. System packages and a virtual screen
import os, subprocess, time, sys, shutil
IN_COLAB = "google.colab" in sys.modules
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)
ROOT = os.geteuid() == 0
SUDO = "" if ROOT else "sudo "          # on WSL / your own PC you are not root
OPT = "/opt" if ROOT else os.path.expanduser("~/bench_opt")
os.makedirs(OPT, exist_ok=True)
sh(f"{SUDO}apt-get -qq update")
r = sh(f"{SUDO}DEBIAN_FRONTEND=noninteractive apt-get -qq install -y xvfb libgl1 libegl1 libgles2 libudev1 libfontconfig1 libxkbcommon0 libxkbcommon-x11-0 libdbus-1-3 libgtk-3-0 libasound2t64 libnss3 libxss1 libgbm1 fonts-liberation libwebkit2gtk-4.1-0 xdotool libgstreamer1.0-0 libgstreamer-plugins-base1.0-0 libgstreamer-plugins-bad1.0-0 libgstreamer-gl1.0-0 gstreamer1.0-plugins-base gstreamer1.0-plugins-good || {SUDO}DEBIAN_FRONTEND=noninteractive apt-get -qq install -y xvfb libgl1 libegl1 libgles2 libudev1 libfontconfig1 libxkbcommon0 libxkbcommon-x11-0 libdbus-1-3 libgtk-3-0 libasound2 libnss3 libxss1 libgbm1 fonts-liberation libwebkit2gtk-4.1-0 xdotool libgstreamer1.0-0 libgstreamer-plugins-base1.0-0 libgstreamer-plugins-bad1.0-0 libgstreamer-gl1.0-0 gstreamer1.0-plugins-base gstreamer1.0-plugins-good")
print("packages:", "ok" if r.returncode == 0 else r.stderr[-300:])
sh(f"{sys.executable} -m pip install -q psutil pandas matplotlib")

os.environ["DISPLAY"] = ":99"
subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1366x768x24"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print("virtual screen ready on", os.environ["DISPLAY"])

In [ ]:
# 2. Settings: edit this cell
RUNS       = 3        # runs per browser and page; the median is reported
SETTLE_S   = 20       # seconds to wait after launch before the final measurement
INTERVAL_S = 1.0
GITHUB_REPO = "Sm0r3s-source/pteron"
PTERON_URL  = None    # None = newest release's linux64.tar.gz; or a direct link
PTERON_PATH = None    # or the path of a pteron you built yourself (folder with resources/ next to it)
# Official Servo builds, tried in order. v0.7.0 is the engine version Pteron is built on.
SERVO_URLS  = ["https://github.com/servo/servo/releases/download/v0.7.0/servo-x86_64-linux-gnu.tar.gz",
               "https://github.com/servo/servo-nightly-builds/releases/latest/download/servo-x86_64-linux-gnu.tar.gz",
               "https://download.servo.org/nightly/linux/servo-latest.tar.gz"]
SERVO_PATH  = None    # or the path of a servoshell you uploaded

In [ ]:
# 3. Test machine (goes into the report)
import platform, json, datetime as dt
def first(cmd, default="?"):
    out = sh(cmd).stdout.strip()
    return out.splitlines()[0].strip() if out else default
mem_gb = round(int(first("awk '/MemTotal/ {print $2}' /proc/meminfo", "0")) / 1048576, 1)
platform_name = "Colab" if os.path.exists("/content") and IN_COLAB else ("Kaggle" if os.path.exists("/kaggle") else ("WSL2" if "microsoft" in platform.release().lower() else "Linux machine"))
SPECS = {
    "platform": platform_name,
    "cpu": first("lscpu | sed -n 's/^Model name:[ ]*//p'"),
    "cores": os.cpu_count(),
    "ram_gb": mem_gb,
    "gpu": "none (software rendering, virtual screen)",
    "os": first("grep PRETTY_NAME /etc/os-release | cut -d= -f2 | tr -d '\"'"),
    "kernel": platform.release(),
    "glibc": first("ldd --version | head -1"),
    "python": platform.python_version(),
    "display": "Xvfb 1366x768x24",
    "date": dt.datetime.now().strftime("%Y-%m-%d %H:%M"),
}
Path = __import__("pathlib").Path
Path("specs.json").write_text(json.dumps(SPECS, indent=2))
for k, v in SPECS.items(): print(f"{k:9}: {v}")

In [ ]:
# 4. Download the browsers (each one is optional; a failure only drops that browser)
import glob, hashlib, re, urllib.request
from pathlib import Path

found = {}      # name -> {"kind", "path", "category", "version"}

def fix_libs(path):
    """Missing system libraries (GStreamer etc.): install them, then say what is still missing."""
    if not sh(f"ldd {path} | grep 'not found'").stdout.strip():
        return ""
    sh(f"{SUDO}DEBIAN_FRONTEND=noninteractive apt-get -qq install -y libgstreamer1.0-0 libgstreamer-plugins-base1.0-0 libgstreamer-plugins-bad1.0-0 libgstreamer-gl1.0-0 gstreamer1.0-plugins-base gstreamer1.0-plugins-good")
    return sh(f"ldd {path} | grep 'not found'").stdout.strip()

# ---- Pteron (the real Linux build from your release) ----
pteron = PTERON_PATH
if not pteron:
    url = PTERON_URL
    if not url:
        api = json.load(urllib.request.urlopen(f"https://api.github.com/repos/{GITHUB_REPO}/releases/latest"))
        url = next((a["browser_download_url"] for a in api["assets"] if a["name"].endswith("linux64.tar.gz")), None)
    if url:
        PT_DIR = f"{OPT}/pteron"; os.makedirs(PT_DIR, exist_ok=True)
        r = sh(f"cd {PT_DIR} && wget -q -O p.tgz '{url}' && tar xzf p.tgz")
        if os.access(f"{PT_DIR}/pteron", os.X_OK): pteron = f"{PT_DIR}/pteron"
        else: print("Pteron download failed:", r.stderr[-300:])
if not pteron: print("Pteron: no linux64 package found in the release.")
if pteron:
    missing = fix_libs(pteron)
    glibc_ok = sh(f"{pteron} --version 2>&1 | head -3")
    if missing or "GLIBC" in glibc_ok.stdout + glibc_ok.stderr:
        print("This machine cannot run Pteron's Linux build:\n", missing or glibc_ok.stdout[:300],
              "\nUse an Ubuntu 24.04 machine (or WSL2), or build Pteron on this machine and set PTERON_PATH.")
    else:
        tag = re.search(r"build-(\d+)", url or "")
        found["pteron"] = {"kind": "pteron", "path": pteron, "category": "Servo engine", "version": f"build {tag.group(1)}" if tag else "custom build"}

# ---- servoshell (the official Servo program; its file is called "servoshell") ----
servo = SERVO_PATH
if not servo:
    SV_DIR = f"{OPT}/servo"; os.makedirs(SV_DIR, exist_ok=True)
    for url in SERVO_URLS:
        r = sh(f"cd {SV_DIR} && rm -rf servo && wget -q -O servo.tar.gz '{url}' && tar xzf servo.tar.gz")
        hits = [p for p in glob.glob(f"{SV_DIR}/**/servoshell", recursive=True) + glob.glob(f"{SV_DIR}/**/servo", recursive=True)
                if os.access(p, os.X_OK) and Path(p).is_file()]
        if hits:
            servo, used = hits[0], url; print("servoshell from", url); break
        print("  servoshell download did not work from", url, r.stderr[-120:].strip())
if servo:
    tag = re.search(r"/(v\d+\.\d+\.\d+)/", used if not SERVO_PATH else "")
    found["servoshell"] = {"kind": "servo", "path": servo, "category": "Servo engine",
                           "version": tag.group(1) if tag else ("nightly" if not SERVO_PATH else "your build")}
    gone = fix_libs(servo)
    if gone: print("servoshell is missing system libraries:\n" + gone)
else:
    print("servoshell not found: upload a Servo build and set SERVO_PATH in cell 2.")

# ---- Firefox ----
FF_DIR = f"{OPT}/ff"
r = sh(f"mkdir -p {FF_DIR} && cd {FF_DIR} && wget -q -O ff.tar.xz 'https://download.mozilla.org/?product=firefox-latest-ssl&os=linux64&lang=en-US' && tar xf ff.tar.xz")
if Path(f"{FF_DIR}/firefox/firefox").exists():
    found["firefox"] = {"kind": "firefox", "path": f"{FF_DIR}/firefox/firefox", "category": "Firefox (Gecko)"}
else: print("Firefox failed:", r.stderr[-200:])

# ---- Chrome ----
r = sh(f"cd /tmp && wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb && {SUDO}DEBIAN_FRONTEND=noninteractive apt-get -qq install -y ./google-chrome-stable_current_amd64.deb")
chrome = shutil.which("google-chrome") or shutil.which("google-chrome-stable")
if chrome: found["chrome"] = {"kind": "chromium", "path": chrome, "category": "Chromium"}
else: print("Chrome failed:", r.stderr[-200:])

BROWSERS = found
for name, cfg in BROWSERS.items():
    if "version" not in cfg:
        cfg["version"] = "nightly" if cfg["kind"] == "servo" else first(f"{cfg['path']} --version 2>/dev/null | head -1", "")[:60]
for n in ("pteron", "servoshell", "firefox", "chrome"):
    c = BROWSERS.get(n)
    print(("OK      " if c else "MISSING ") + f"{n:11}", f"[{c['category']}] {c['version']}" if c else "")

In [ ]:
# 5. Test pages
PAGES = [
    ("blank", "{fx}/blank.html"), ("text", "{fx}/text.html"), ("table", "{fx}/table.html"),
    ("css", "{fx}/css.html"), ("js", "{fx}/js.html"),
    # Real sites (Colab's network is fast and shared; comment out any you don't want)
    ("example",    "https://example.com"),
    ("wikipedia",  "https://en.wikipedia.org/wiki/Rust_(programming_language)"),
    ("hackernews", "https://news.ycombinator.com"),
    ("github",     "https://github.com/servo/servo"),
]

# Local test pages (same for every browser, no network noise)
import threading
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer

FIXTURES = Path("fixtures"); FIXTURES.mkdir(exist_ok=True)
FX_PORT = 8765
FX_URL = f"http://127.0.0.1:{FX_PORT}"

(FIXTURES / "blank.html").write_text("<!doctype html><title>blank</title><body></body>")
words = "lorem ipsum dolor sit amet consectetur adipiscing elit sed do eiusmod".split()
paras = "\n".join(f"<p>{' '.join(words[(i + j) % len(words)] for j in range(60))}</p>" for i in range(3000))
(FIXTURES / "text.html").write_text(f"<!doctype html><title>text</title><body>{paras}</body>")
rows = "".join("<tr>" + "".join(f"<td>r{r}c{c}</td>" for c in range(20)) + "</tr>" for r in range(300))
(FIXTURES / "table.html").write_text(f"<!doctype html><title>table</title><body><table>{rows}</table></body>")
boxes = "".join(f'<div style="display:inline-block;width:90px;height:60px;margin:4px;border-radius:10px;'
                f'box-shadow:0 2px 8px rgba(0,0,0,.4);background:linear-gradient({i % 360}deg,#08f,#f80)">{i}</div>'
                for i in range(2000))
(FIXTURES / "css.html").write_text(f"<!doctype html><title>css</title><body>{boxes}</body>")
(FIXTURES / "js.html").write_text(
    "<!doctype html><title>js</title><body><ul id=l></ul><script>"
    "const l=document.getElementById('l');"
    "for(let i=0;i<20000;i++){const li=document.createElement('li');li.textContent='item '+i;l.appendChild(li);}"
    "window.big=Array.from({length:1000000},(_, i)=>({i,s:'x'+i}));</script></body>")

handler = partial(SimpleHTTPRequestHandler, directory=str(FIXTURES))
handler.log_message = lambda *a, **k: None
try:
    server = ThreadingHTTPServer(("127.0.0.1", FX_PORT), handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    print("fixture server on", FX_URL)
except OSError:
    print("fixture server already running on", FX_URL, "(fine when re-running this cell)")

In [ ]:
# 6. Measuring code
import csv, random, statistics, subprocess, tempfile, time
from datetime import datetime
import psutil

MB = 1024 * 1024
KIND_ARGS = {
    "chromium": ["--user-data-dir={profile}", "--no-first-run", "--no-default-browser-check", "--disable-sync",
                 "--no-sandbox", "--disable-dev-shm-usage", "{url}"],   # Colab runs as root, so Chrome needs --no-sandbox
    "firefox":  ["--no-remote", "--new-instance", "--profile", "{profile}", "{url}"],
    "servo":    ["{url}"],
    "pteron":   ["{url}"],
}
FIREFOX_PREFS = '''
user_pref("browser.shell.checkDefaultBrowser", false);
user_pref("browser.aboutwelcome.enabled", false);
user_pref("browser.startup.homepage_override.mstone", "ignore");
user_pref("startup.homepage_welcome_url", "");
user_pref("startup.homepage_welcome_url.additional", "");
user_pref("datareporting.policy.dataSubmissionEnabled", false);
user_pref("app.update.auto", false);
user_pref("app.update.enabled", false);
'''

def collect(root_pid, profile):
    '''The launched process, all its children, and any process whose command line names the temp profile.'''
    found = {}
    try:
        root = psutil.Process(root_pid)
        for p in [root] + root.children(recursive=True):
            found[p.pid] = p
    except psutil.NoSuchProcess:
        pass
    key = profile.lower()
    for p in psutil.process_iter(["cmdline"]):
        try:
            if key in " ".join(p.info["cmdline"] or []).lower():
                found[p.pid] = p
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return list(found.values())

def sample(procs):
    uss = rss = private = n = 0
    for p in procs:
        try:
            uss += p.memory_full_info().uss
            info = p.memory_info()
            rss += info.rss
            private += getattr(info, "private", info.rss)   # Windows: private (committed) bytes
            n += 1
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return uss, rss, private, n

def kill_all(procs):
    for p in procs:
        try: p.terminate()
        except psutil.Error: pass
    _, alive = psutil.wait_procs(procs, timeout=5)
    for p in alive:
        try: p.kill()
        except psutil.Error: pass

def run_one(name, cfg, url):
    kind = cfg.get("kind", "generic")
    profile = tempfile.mkdtemp(prefix=f"bench-{name}-")
    if kind == "firefox":
        (Path(profile) / "user.js").write_text(FIREFOX_PREFS)
    args = [a.format(url=url, profile=profile) for a in (cfg.get("args") or KIND_ARGS.get(kind, ["{url}"]))]
    env = dict(os.environ)
    if kind == "pteron":   # fresh profile for programs that keep their data in the XDG folders
        env["XDG_CONFIG_HOME"] = env["XDG_DATA_HOME"] = env["XDG_CACHE_HOME"] = profile
    proc = subprocess.Popen([cfg["path"]] + args, stdout=open(Path(profile) / 'browser.log', 'w'), stderr=subprocess.STDOUT, env=env,
                            cwd=os.path.dirname(cfg["path"]) or None)
    series, seen, t0 = [], {}, time.time()
    try:
        while time.time() - t0 < SETTLE_S:
            time.sleep(INTERVAL_S)
            procs = collect(proc.pid, profile)
            seen.update({p.pid: p for p in procs})
            series.append(sample(procs))
    finally:
        kill_all(list(seen.values()))
        time.sleep(1.5)
        lp = Path(profile) / 'browser.log'
        tail_log = ' | '.join(lp.read_text(errors='ignore').strip().splitlines()[-4:])[:300] if lp.exists() else ''
        shutil.rmtree(profile, ignore_errors=True)
    series = [s for s in series if s[3] > 0]
    if not series:
        raise RuntimeError("no browser processes were found. Its output: " + tail_log)
    tail = series[-5:]
    if max(s[0] for s in tail) < 20 * MB:
        raise RuntimeError('the browser closed itself or never started. Its output: ' + tail_log)
    med = lambda i: round(statistics.median(s[i] for s in tail) / MB, 1)
    return {"uss_mb": med(0), "private_mb": med(2), "rss_mb": med(1),
            "uss_peak_mb": round(max(s[0] for s in series) / MB, 1),
            "procs": max(s[3] for s in series)}

In [ ]:
# 7. Run (takes a while: browsers x pages x runs x settle time). Results are appended to results.csv
RESULTS = Path("results.csv")
FIELDS = ["timestamp", "browser", "page", "run", "settle_s", "uss_mb", "private_mb", "rss_mb", "uss_peak_mb", "procs"]
pages = [(n, u.replace("{fx}", FX_URL)) for n, u in PAGES]
total = len(BROWSERS) * len(pages) * RUNS
done = 0
if not BROWSERS:
    raise SystemExit("No browsers found. Fix the paths in cell 2.")
new = not RESULTS.exists()
with RESULTS.open("a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    if new: w.writeheader()
    for run in range(1, RUNS + 1):
        for page, url in pages:
            order = list(BROWSERS.items()); random.shuffle(order)   # no browser always goes first
            for name, cfg in order:
                done += 1
                print(f"[{done}/{total}] {name} :: {page} (run {run}) ...", end=" ", flush=True)
                try:
                    r = run_one(name, cfg, url)
                except Exception as e:
                    print(f"FAILED ({e})"); continue
                print(f"{r['private_mb']} MB private, {r['uss_mb']} MB uss, {r['procs']} procs")
                w.writerow({"timestamp": datetime.now().isoformat(timespec="seconds"), "browser": name,
                            "page": page, "run": run, "settle_s": SETTLE_S, **r})
                f.flush()
                time.sleep(3)
print("Done. Results in", RESULTS.resolve())

In [ ]:
# 7b. Ten tabs open at once (shows tab handling and Pteron's sleeping tabs; takes about 8 minutes per browser)
import urllib.request, json as _json
TABS_AT_S   = (60, 420)     # measure after 1 minute and after 7 minutes (Pteron's tabs sleep after 5 minutes of not being used)
TAB_URLS = [f"{FX_URL}/{p}.html?t={i}" for i, p in enumerate(["text", "table", "css", "js", "blank"])] + [
    "https://example.com", "https://en.wikipedia.org/wiki/Rust_(programming_language)",
    "https://news.ycombinator.com", "https://github.com/servo/servo", f"{FX_URL}/text.html?t=9"]
WD_PORT = 17333      # unusual on purpose: Kaggle and Colab already use some low ports
lab = lambda at: f"{at // 60} min" if at % 60 == 0 else f"{at} s"

def drive_tabs(urls, proc, log_tail):
    """Servo-based browsers: open the other tabs through their WebDriver port (no keyboard needed)."""
    base = f"http://127.0.0.1:{WD_PORT}"
    def call(method, path, data=None):
        req = urllib.request.Request(base + path, _json.dumps(data).encode() if data is not None else None,
                                     {"content-type": "application/json"}, method=method)
        return _json.load(urllib.request.urlopen(req, timeout=90))["value"]
    last = None
    for _ in range(60):                       # up to about 2 minutes: software rendering starts slowly
        if proc.poll() is not None:
            raise RuntimeError("the browser exited at start: " + log_tail())
        try:
            sid = call("POST", "/session", {"capabilities": {}})["sessionId"]; break
        except Exception as e:
            last = e; time.sleep(2)
    else:
        raise RuntimeError(f"WebDriver did not answer ({str(last)[:80]}). Browser log: " + log_tail())
    for url in urls[1:]:
        try:
            handle = call("POST", f"/session/{sid}/window/new", {"type": "tab"})["handle"]
            call("POST", f"/session/{sid}/window", {"handle": handle})
            call("POST", f"/session/{sid}/url", {"url": url})
        except Exception as e:
            print("  (a tab did not load:", str(e)[:60] + ")", end=" ")

def run_tabs(name, cfg, urls):
    kind = cfg["kind"]
    profile = tempfile.mkdtemp(prefix=f"bench-{name}-")
    if kind == "firefox":
        (Path(profile) / "user.js").write_text(FIREFOX_PREFS)
    env = dict(os.environ)
    if kind == "pteron":
        env["XDG_CONFIG_HOME"] = env["XDG_DATA_HOME"] = env["XDG_CACHE_HOME"] = profile
    template = KIND_ARGS.get(kind, ["{url}"])
    if kind in ("pteron", "servo"):
        args = [f"--webdriver={WD_PORT}", urls[0]]
    else:
        args = [a.format(profile=profile) for a in template if a != "{url}"] + urls   # one command line, many tabs
    logfile = Path(profile) / "browser.log"
    log_tail = lambda: " | ".join(logfile.read_text(errors="ignore").strip().splitlines()[-4:])[:300] if logfile.exists() else ""
    proc = subprocess.Popen([cfg["path"]] + args, stdout=open(logfile, "w"), stderr=subprocess.STDOUT, env=env,
                            cwd=os.path.dirname(cfg["path"]) or None)
    series, seen, t0 = [], {}, time.time()
    try:
        if kind in ("pteron", "servo"):
            drive_tabs(urls, proc, log_tail)
            t0 = time.time()      # the clock starts when all tabs are open
        while time.time() - t0 < max(TABS_AT_S) + 5:
            time.sleep(INTERVAL_S)
            procs = collect(proc.pid, profile)
            seen.update({p.pid: p for p in procs})
            series.append((time.time() - t0, *sample(procs)))
    finally:
        kill_all(list(seen.values())); time.sleep(1.5); shutil.rmtree(profile, ignore_errors=True)
    out = {}
    for at in TABS_AT_S:
        win = [s for s in series if abs(s[0] - at) <= 5 and s[4] > 0]
        if not win: raise RuntimeError("no measurement at %d s" % at)
        med = lambda i: round(statistics.median(s[i] for s in win) / MB, 1)
        out[at] = {"uss_mb": med(1), "private_mb": med(3), "rss_mb": med(2), "uss_peak_mb": round(max(s[1] for s in series) / MB, 1), "procs": max(s[4] for s in win)}
    return out

new = not RESULTS.exists()
with RESULTS.open("a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    if new: w.writeheader()
    for name, cfg in BROWSERS.items():
        print(f"10 tabs :: {name} ...", end=" ", flush=True)
        try:
            res = run_tabs(name, cfg, TAB_URLS)
        except Exception as e:
            print(f"FAILED ({e})"); continue
        print(", ".join(f"{res[a]['uss_mb']} MB at {lab(a)}" for a in TABS_AT_S))
        for at, r in res.items():
            w.writerow({"timestamp": datetime.now().isoformat(timespec="seconds"), "browser": name,
                        "page": f"10 tabs @{lab(at)}", "run": 1, "settle_s": at, **r})
        f.flush(); time.sleep(3)

In [ ]:
# 8. Report: test machine, then the browsers grouped by category
import pandas as pd, json
import matplotlib.pyplot as plt
SPECS = json.load(open("specs.json"))
df_all = pd.read_csv("results.csv")
is_tabs = df_all.page.str.startswith("10 tabs")
df, dft = df_all[~is_tabs].copy(), df_all[is_tabs].copy()
METRIC = "uss_mb"      # private memory in RAM
CAT = {n: c["category"] for n, c in BROWSERS.items()}
VER = {n: c.get("version", "") for n, c in BROWSERS.items()}
df["category"] = df.browser.map(CAT)
BASELINE = next((b for b in ("pteron", "servoshell") if b in set(df.browser)), df.browser.iloc[0])
CAT_ORDER = ["Servo engine", "Firefox (Gecko)", "Chromium"]
order_pages = [p for p, _ in PAGES if p in set(df.page)]

lines = ["# Browser memory benchmark", "", "## Test machine", ""]
lines += [f"- **Platform:** {SPECS['platform']}", f"- **CPU:** {SPECS['cpu']} ({SPECS['cores']} cores)",
          f"- **RAM:** {SPECS['ram_gb']} GB", f"- **GPU:** {SPECS['gpu']}", f"- **OS:** {SPECS['os']}, kernel {SPECS['kernel']}",
          f"- **Display:** {SPECS['display']}", f"- **Date:** {SPECS['date']}",
          f"- **Method:** whole process tree, {METRIC}, median of {df.run.max()} runs, {int(df.settle_s.iloc[0])} s after launch", ""]

per_browser = (df.groupby(["category", "browser"])[METRIC].median().rename("median_mb").reset_index())
per_browser["x_" + BASELINE] = (per_browser.median_mb / per_browser.loc[per_browser.browser == BASELINE, "median_mb"].iloc[0]).round(2)
per_browser["version"] = per_browser.browser.map(VER)
per_browser["cat_i"] = per_browser.category.map(lambda c: CAT_ORDER.index(c) if c in CAT_ORDER else 99)
per_browser = per_browser.sort_values(["cat_i", "median_mb"]).drop(columns="cat_i")
per_browser["median_mb"] = per_browser.median_mb.round(0)

lines += ["## Results by category (median over all pages, lower is better)", ""]
for cat, grp in per_browser.groupby("category", sort=False):
    lines += [f"### {cat}", "", "| Browser | Version | Memory (MB) | vs " + BASELINE + " |", "|---|---|---|---|"]
    lines += [f"| {r.browser} | {r.version} | {int(r.median_mb)} | {r['x_' + BASELINE]}x |" for _, r in grp.iterrows()] + [""]

tab = df.pivot_table(index="page", columns="browser", values=METRIC, aggfunc="median").reindex(order_pages)
tab = tab[[b for b in per_browser.browser]]
lines += ["## Per page (MB)", "", "| page | " + " | ".join(tab.columns) + " |", "|---|" + "---|" * len(tab.columns)]
lines += [f"| {p} | " + " | ".join("" if pd.isna(v) else str(int(round(v))) for v in row) + " |" for p, row in tab.iterrows()]
if len(dft):
    dft["category"] = dft.browser.map(CAT)
    tt = dft.pivot_table(index=["category", "browser"], columns="page", values=METRIC, aggfunc="median").reset_index()
    secs = lambda c: int(c.split("@")[1].split()[0]) * (60 if c.endswith("min") else 1)
    cols = sorted([c for c in tt.columns if str(c).startswith("10 tabs")], key=secs)
    tt["cat_i"] = tt.category.map(lambda c: CAT_ORDER.index(c) if c in CAT_ORDER else 99)
    tt = tt.sort_values(["cat_i", cols[-1]])
    base = tt.loc[tt.browser == BASELINE, cols[-1]]
    lines += ["", "## Ten tabs open (MB; the last column is after the longest wait, when sleeping tabs have had time to sleep)", "",
              "| Category | Browser | " + " | ".join(cols) + " | vs " + BASELINE + " (last) |", "|---|---|" + "---|" * (len(cols) + 1)]
    for _, r in tt.iterrows():
        ratio = f"{r[cols[-1]] / base.iloc[0]:.2f}x" if len(base) else ""
        lines.append(f"| {r.category} | {r.browser} | " + " | ".join(str(int(round(r[c]))) for c in cols) + f" | {ratio} |")
lines += ["", "Single-page tests favour small engines; Servo renders some sites worse than Firefox and Chrome. Compare within this run only."]
open("report.md", "w").write("\n".join(lines))
print("\n".join(lines))

colors = {"Servo engine": "#e8590c", "Firefox (Gecko)": "#7048e8", "Chromium": "#1971c2"}
fig, ax = plt.subplots(figsize=(10, 0.7 * len(per_browser) + 2.6))
rows = per_browser.iloc[::-1]
ax.barh(rows.browser, rows.median_mb, color=[colors.get(c, "#868e96") for c in rows.category])
for y, (v, c) in enumerate(zip(rows.median_mb, rows.category)):
    ax.text(v, y, f" {int(v)} MB", va="center", fontsize=9)
ax.set_xlim(0, rows.median_mb.max() * 1.15)
ax.set_xlabel("Memory after load, whole process tree (MB, lower is better)")
ax.set_title(f"Browser memory by category: {SPECS['cpu']}, {SPECS['cores']} cores, {SPECS['ram_gb']} GB RAM, {SPECS['platform']}", fontsize=10)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=colors[c]) for c in CAT_ORDER if c in set(rows.category)],
          labels=[c for c in CAT_ORDER if c in set(rows.category)], loc="lower right")
ax.grid(axis="x", alpha=0.3); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig("benchmark_linux.png", dpi=160); plt.show()

In [ ]:
# 9. Get the files (results.csv, report.md, specs.json, benchmark_linux.png)
files_out = ["results.csv", "report.md", "specs.json", "benchmark_linux.png"]
try:
    from google.colab import files
    for f in files_out: files.download(f)
except ImportError:
    print("Files are in", os.getcwd(), ":", ", ".join(files_out), "(on Kaggle: they are in the Output tab)")

## Reading the result honestly

- Pteron here is its **Linux build** under a virtual screen without a GPU. Windows numbers (`pteron_benchmark.ipynb`) will differ.
- Single-page tests favour small engines. Servo renders some sites worse than Firefox and Chrome; say that next to the chart.
- Chrome runs with `--no-sandbox` (needed as root) and no GPU. That is the same for every browser, but it is not how people run them.
- Publish `report.md` and `results.csv` together so others can check the numbers.